# 02. 합성 데이터 설계 — 4축 구조 분석 & 생성 가이드라인

**목적**: 위협 대화의 구조적 특성을 수치로 파악하고, 그 "반대 설계"로 합성 일반 대화 생성 가이드라인을 도출

**4축 분석 프레임워크**:
1. 발화 길이 대칭성 (Tiki-taka vs 비대칭)
2. 존댓말/반말 비대칭 (권력 관계)
3. 감정 에스칼레이션 (전반 vs 후반)
4. 굴복 표현 vs 대등 표현

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 5)

# 데이터 로드
train = pd.read_csv('aiffel-d-lthon-dktc-online-17/train.csv')
train = train.drop_duplicates(subset='conversation').reset_index(drop=True)
print(f'Train: {len(train)}건')
print(train['class'].value_counts())

## 축 1: 발화 길이 대칭성

> 위협 대화는 **가해자가 길게 말하고 피해자가 짧게 반응**하는 비대칭 구조.
> 일반 대화는 **서로 비슷한 길이로 주고받는** 대칭 구조여야 함.

In [ ]:
def turn_asymmetry(text):
    """홀수턴(화자A) vs 짝수턴(화자B) 평균 길이 비율. 1.0 = 완전 대칭"""
    turns = [t.strip() for t in str(text).split('\n') if t.strip()]
    if len(turns) < 2:
        return None
    odd_lens = [len(turns[i]) for i in range(0, len(turns), 2)]
    even_lens = [len(turns[i]) for i in range(1, len(turns), 2)]
    if not even_lens:
        return None
    a_mean, b_mean = np.mean(odd_lens), np.mean(even_lens)
    if min(a_mean, b_mean) == 0:
        return None
    return max(a_mean, b_mean) / min(a_mean, b_mean)

train['turn_asym'] = train['conversation'].apply(turn_asymmetry)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 분포
for cls in train['class'].unique():
    subset = train[train['class'] == cls]['turn_asym'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.5, label=cls, range=(1, 4))
axes[0].axvline(x=1.5, color='red', linestyle='--', alpha=0.7, label='비대칭 기준 (1.5)')
axes[0].set_title('클래스별 발화 길이 대칭비 분포', fontweight='bold')
axes[0].set_xlabel('대칭비 (1.0=완전대칭)')
axes[0].legend(fontsize=8)

# 요약 통계
summary = train.groupby('class')['turn_asym'].agg(['mean', 'median']).round(2)
summary['비대칭(>1.5)%'] = train.groupby('class')['turn_asym'].apply(lambda x: (x > 1.5).mean() * 100).round(1)
print(summary)

# 바 차트
summary['비대칭(>1.5)%'].plot(kind='bar', ax=axes[1], color=['#f59e0b', '#8b5cf6', '#3b82f6', '#ef4444'])
axes[1].set_title('클래스별 비대칭 대화 비율 (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].axhline(y=20, color='green', linestyle='--', alpha=0.7, label='일반 대화 목표 (<20%)')
axes[1].legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('figures/06_turn_asymmetry.png', dpi=150, bbox_inches='tight')
plt.show()

## 축 2: 존댓말/반말 비대칭 (권력 관계)

> 위협 대화는 **한쪽이 존대하고 다른 쪽이 반말**하는 권력 비대칭이 뚜렷.
> 일반 대화는 **양쪽이 같은 화법**을 사용해야 함 (둘 다 반말 또는 둘 다 존대).

In [ ]:
POLITE = ['습니다', '세요', '에요', '해요', '하세요', '입니다', '합니다', '까요', '나요', '죠']
CASUAL = ['해', '야', '어', '지', '냐', '라', '거든', '잖아', '는데']

def speech_asymmetry(text):
    """홀수턴 vs 짝수턴의 존댓말 비율 차이. 0 = 대칭, 1 = 극단 비대칭"""
    turns = [t.strip() for t in str(text).split('\n') if t.strip()]
    if len(turns) < 4:
        return None
    def polite_ratio(turn_list):
        polite = sum(1 for t in turn_list for e in POLITE if t.endswith(e))
        casual = sum(1 for t in turn_list for e in CASUAL if t.endswith(e))
        total = polite + casual
        return polite / total if total > 0 else 0.5
    odd = [turns[i] for i in range(0, len(turns), 2)]
    even = [turns[i] for i in range(1, len(turns), 2)]
    return abs(polite_ratio(odd) - polite_ratio(even))

train['speech_asym'] = train['conversation'].apply(speech_asymmetry)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in train['class'].unique():
    subset = train[train['class'] == cls]['speech_asym'].dropna()
    axes[0].hist(subset, bins=20, alpha=0.5, label=cls, range=(0, 1))
axes[0].axvline(x=0.3, color='red', linestyle='--', alpha=0.7, label='강한 비대칭 기준')
axes[0].set_title('클래스별 존댓말 비대칭 분포', fontweight='bold')
axes[0].set_xlabel('비대칭도 (0=대칭)')
axes[0].legend(fontsize=8)

summary2 = train.groupby('class')['speech_asym'].agg(['mean', 'median']).round(3)
summary2['강한비대칭(>0.3)%'] = train.groupby('class')['speech_asym'].apply(lambda x: (x > 0.3).mean() * 100).round(1)
print(summary2)

summary2['강한비대칭(>0.3)%'].plot(kind='bar', ax=axes[1], color=['#f59e0b', '#8b5cf6', '#3b82f6', '#ef4444'])
axes[1].set_title('클래스별 존댓말 비대칭 비율 (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].axhline(y=10, color='green', linestyle='--', alpha=0.7, label='일반 대화 목표 (<10%)')
axes[1].legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('figures/07_speech_asymmetry.png', dpi=150, bbox_inches='tight')
plt.show()

## 축 3: 감정 에스칼레이션

> 위협 대화는 **후반부로 갈수록 위협 단어 밀도가 급증** (협박: 59.3%가 에스컬레이션).
> 일반 대화는 **전반~후반 감정 밀도가 일정** (Flat Sentiment)해야 함.

In [ ]:
THREAT_WORDS = ['죽', '살려', '칼', '때려', '패', '돈', '내놔', '죄송', '용서', '제발',
                '죽여', '시키는', '맞을', '꺼져', '닥쳐', '못생']

def escalation_ratio(text):
    """후반부 50% vs 전반부 50%의 위협 단어 밀도 차이"""
    turns = [t.strip() for t in str(text).split('\n') if t.strip()]
    if len(turns) < 4:
        return None
    mid = len(turns) // 2
    first_half = ' '.join(turns[:mid])
    second_half = ' '.join(turns[mid:])
    def threat_density(part):
        words = part.split()
        if not words: return 0
        return sum(1 for w in words for tw in THREAT_WORDS if tw in w) / len(words)
    return threat_density(second_half) - threat_density(first_half)

train['escalation'] = train['conversation'].apply(escalation_ratio)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in train['class'].unique():
    subset = train[train['class'] == cls]['escalation'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.5, label=cls, range=(-0.15, 0.15))
axes[0].axvline(x=0, color='gray', linestyle='-', alpha=0.5)
axes[0].set_title('클래스별 에스칼레이션 분포', fontweight='bold')
axes[0].set_xlabel('후반-전반 위협밀도 차이 (양수=에스컬레이션)')
axes[0].legend(fontsize=8)

summary3 = pd.DataFrame()
for cls in train['class'].unique():
    esc = train[train['class'] == cls]['escalation'].dropna()
    summary3.loc[cls, '에스컬(>0)%'] = (esc > 0).mean() * 100
    summary3.loc[cls, '평탄(=0)%'] = (esc == 0).mean() * 100
    summary3.loc[cls, '디에스컬(<0)%'] = (esc < 0).mean() * 100
print(summary3.round(1))

summary3[['에스컬(>0)%', '디에스컬(<0)%']].plot(kind='bar', ax=axes[1])
axes[1].set_title('에스컬레이션 vs 디에스컬레이션 비율', fontweight='bold')
axes[1].set_ylabel('%')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('figures/08_escalation.png', dpi=150, bbox_inches='tight')
plt.show()

## 축 4: 굴복 표현 vs 대등 표현

> 위협 대화의 피해자는 "죄송합니다", "잘못했습니다", "제발"로 **굴복**.
> 일반 대화에서는 "맞아", "좋아", "인정"처럼 **대등하게** 반응.

In [ ]:
SUBMISSIVE = ['죄송합니다', '잘못했습니다', '살려주세요', '제발', '용서', '그만', '안됩니다', '못합니다']
EQUAL = ['맞아', '그래', '좋아', '오케이', '인정', '대박', '진짜', '그러게', '나도']

train['submissive'] = train['conversation'].apply(lambda x: sum(1 for s in SUBMISSIVE if s in str(x)))
train['equal'] = train['conversation'].apply(lambda x: sum(1 for s in EQUAL if s in str(x)))
train['sub_eq_ratio'] = train['submissive'] / (train['equal'] + 0.01)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 클래스별 굴복/대등 평균
summary4 = train.groupby('class')[['submissive', 'equal']].mean().round(2)
summary4.plot(kind='bar', ax=axes[0])
axes[0].set_title('클래스별 굴복/대등 표현 평균 빈도', fontweight='bold')
axes[0].set_ylabel('평균 출현 횟수')
plt.sca(axes[0])
plt.xticks(rotation=15)

# 비율
ratio = train.groupby('class')['sub_eq_ratio'].mean().round(2)
ratio.plot(kind='bar', ax=axes[1], color=['#f59e0b', '#8b5cf6', '#3b82f6', '#ef4444'])
axes[1].set_title('굴복/대등 비율 (낮을수록 대등)', fontweight='bold')
axes[1].set_ylabel('비율')
axes[1].axhline(y=0.1, color='green', linestyle='--', alpha=0.7, label='일반 대화 목표 (<0.1)')
axes[1].legend()
plt.xticks(rotation=15)

print(summary4)
print(f'\n굴복/대등 비율:\n{ratio}')

plt.tight_layout()
plt.savefig('figures/09_submissive_vs_equal.png', dpi=150, bbox_inches='tight')
plt.show()

## 종합: 합성 일반 대화 생성 가이드라인

| 축 | 위협 대화 프로파일 | 합성 일반 대화 목표 | 검증 기준 |
|---|---|---|---|
| **발화 대칭성** | 비대칭 53~60% (비율 1.5~2.0) | 비대칭 < 20% (비율 1.0~1.2) | `turn_asymmetry() < 1.3` |
| **존댓말 비대칭** | 한쪽만 존대 45~55% | 비대칭 < 10% | `speech_asymmetry() < 0.1` |
| **에스칼레이션** | 후반 증가 59% (협박) | 에스컬 < 20% | `escalation_ratio() ≈ 0` |
| **굴복/대등 비율** | 협박·직장: 1.02 | < 0.1 (굴복 표현 거의 없음) | `submissive < 0.5` |

### 표면층 (기존 strategy_v1.md)
- 이모티콘/초성: 사용 금지 (원본 emoticon_len = 0.0)
- 느낌표: 대화당 0~2개
- 물음표: 대화당 3~5개
- 글자 수: 180~250자
- 발화 횟수: 10턴 (±2)

## 접근 B: 위협 대화 "중화(Neutralize)" 실험

> 위협 대화의 **구조를 그대로 유지**하면서 **위협 요소만 무해한 내용으로 교체**.
> 이렇게 하면 발화 대칭성, 존댓말 패턴, 에스칼레이션 등의 구조가 자연스럽게 보존됨.

In [ ]:
# 위협 대화 → 중화 변환 아이디어 프로토타입
# 실제 구현은 LLM을 활용하여 의미를 변환

# 예시: 협박 대화 원본
example_threat = train[train['class'] == '협박 대화'].iloc[0]['conversation']
print('=== 원본 (협박) ===')
for i, turn in enumerate(example_threat.split('\n')):
    speaker = 'A' if i % 2 == 0 else 'B'
    print(f'  [{speaker}] {turn.strip()}')

print()
print('=== 중화 아이디어 ===')
print('위협 키워드를 무해한 맥락으로 치환하되,')
print('발화 길이 비율, 존댓말 패턴, 감정 곡선은 유지')
print()
print('예시 변환 방향:')
print('  "죽여버리겠다" → "화가 난다" (감정은 유지, 위협은 제거)')
print('  "제발 살려주세요" → "제발 도와주세요" (굴복 구조 유지)')
print()
print('⚠️ 하지만 이 방식은 굴복 구조 자체를 유지하므로')
print('   일반 대화라기보다 "순화된 위협"이 될 위험.')  
print('   → 위협의 "구조"가 아닌 "내용 분포"만 참고하는 것이 더 안전')

## 다음 단계

1. 현재 합성 데이터의 4축 수치 측정 (위협 프로파일과 비교)
2. 4축 가이드라인 반영한 프롬프트 설계
3. 새 합성 데이터 생성 & 4축 검증
4. strategy_v1 세션에서 학습 실험